In [ ]:
# ============================================================
# Download WTMC2021 dataset and inspect CSV schemas
# ============================================================

DATASET_URL = (
    "https://intrusion-detection.distrinet-research.be/"
    "WTMC2021/Dataset/dataset.zip"
)

ZIP_PATH = "/content/wtmc2021_dataset.zip"

# -c: استكمال التنزيل عند الانقطاع
# -O: حفظ الملف باسم ثابت
!wget -c --progress=bar:force:noscroll \
    -O "$ZIP_PATH" \
    "$DATASET_URL"

# التحقق من الملف
!ls -lh "$ZIP_PATH"
!unzip -t "$ZIP_PATH" | tail -n 5

--2026-11-02 21:11:55--  https://intrusion-detection.distrinet-research.be/WTMC2021/Dataset/dataset.zip
Resolving intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)... 134.58.40.205
Connecting to intrusion-detection.distrinet-research.be (intrusion-detection.distrinet-research.be)|134.58.40.205|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 333841436 (318M) [application/zip]
Saving to: ‘/content/wtmc2021_dataset.zip’

/content/wtmc2021_d 100%[===================>] 318.38M  7.66MB/s    in 54s     

2026-11-02 21:12:51 (5.86 MB/s) - ‘/content/wtmc2021_dataset.zip’ saved [333841436/333841436]

-rw-r--r-- 1 root root 319M Nov 24  2021 /content/wtmc2021_dataset.zip
    testing: Monday-WorkingHours.csv   OK
    testing: Thursday-WorkingHours.csv   OK
    testing: Tuesday-WorkingHours.csv   OK
    testing: Wednesday-WorkingHours.csv   OK
No errors detected in compressed data of /content/wtmc2021_dataset.zip.


In [ ]:
import zipfile
import pandas as pd

with zipfile.ZipFile(ZIP_PATH) as archive:

    csv_files = [
        name
        for name in archive.namelist()
        if name.lower().endswith(".csv")
        and not name.endswith("/")
    ]

    print(f"CSV files found: {len(csv_files)}\n")

    schemas = {}

    for filename in csv_files:
        print("=" * 100)
        print(filename)
        print("=" * 100)

        with archive.open(filename) as stream:
            sample = pd.read_csv(
                stream,
                nrows=3,
                low_memory=False,
            )

        schemas[filename] = sample.columns.tolist()

        for index, column in enumerate(
            sample.columns,
            start=1,
        ):
            print(f"{index:3d}. {column}")

CSV files found: 5

Friday-WorkingHours.csv
  1. Flow ID
  2. Src IP
  3. Src Port
  4. Dst IP
  5. Dst Port
  6. Protocol
  7. Timestamp
  8. Flow Duration
  9. Total Fwd Packet
 10. Total Bwd packets
 11. Total Length of Fwd Packet
 12. Total Length of Bwd Packet
 13. Fwd Packet Length Max
 14. Fwd Packet Length Min
 15. Fwd Packet Length Mean
 16. Fwd Packet Length Std
 17. Bwd Packet Length Max
 18. Bwd Packet Length Min
 19. Bwd Packet Length Mean
 20. Bwd Packet Length Std
 21. Flow Bytes/s
 22. Flow Packets/s
 23. Flow IAT Mean
 24. Flow IAT Std
 25. Flow IAT Max
 26. Flow IAT Min
 27. Fwd IAT Total
 28. Fwd IAT Mean
 29. Fwd IAT Std
 30. Fwd IAT Max
 31. Fwd IAT Min
 32. Bwd IAT Total
 33. Bwd IAT Mean
 34. Bwd IAT Std
 35. Bwd IAT Max
 36. Bwd IAT Min
 37. Fwd PSH Flags
 38. Bwd PSH Flags
 39. Fwd URG Flags
 40. Bwd URG Flags
 41. Fwd Header Length
 42. Bwd Header Length
 43. Fwd Packets/s
 44. Bwd Packets/s
 45. Packet Length Min
 46. Packet Length Max
 47. Packet Length Mean

In [ ]:
import re

def normalize(name):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(name).lower(),
    )

required_features = {
    "Timestamp": [
        "Timestamp",
        "Flow Timestamp",
        "DateTime",
        "Start Time",
    ],
    "Source IP": [
        "Src IP",
        "Source IP",
        "srcip",
    ],
    "Destination IP": [
        "Dst IP",
        "Destination IP",
        "dstip",
    ],
    "Flow IAT Max": [
        "Flow IAT Max",
        "Flow_IAT_Max",
    ],
    "Avg Bwd Segment Size": [
        "Avg Bwd Segment Size",
        "Avg_Bwd_Segment_Size",
        "Bwd Segment Size Avg",
    ],
    "Down/Up Ratio": [
        "Down/Up Ratio",
        "Down Up Ratio",
        "Down_Up_Ratio",
    ],
    "Label": [
        "Label",
        "Class",
        "Attack",
        "Attack Label",
    ],
}

all_columns = sorted({
    column
    for columns in schemas.values()
    for column in columns
})

normalized_columns = {
    normalize(column): column
    for column in all_columns
}

results = []

for requirement, aliases in required_features.items():
    matched = None

    for alias in aliases:
        if normalize(alias) in normalized_columns:
            matched = normalized_columns[
                normalize(alias)
            ]
            break

    results.append({
        "Required component": requirement,
        "Status": "FOUND" if matched else "MISSING",
        "Matched column": matched or "",
    })

compatibility = pd.DataFrame(results)

print("\nFROZEN-PROTOCOL COMPATIBILITY")
print("=" * 80)
print(compatibility.to_string(index=False))


FROZEN-PROTOCOL COMPATIBILITY
  Required component Status       Matched column
           Timestamp  FOUND            Timestamp
           Source IP  FOUND               Src IP
      Destination IP  FOUND               Dst IP
        Flow IAT Max  FOUND         Flow IAT Max
Avg Bwd Segment Size  FOUND Bwd Segment Size Avg
       Down/Up Ratio  FOUND        Down/Up Ratio
               Label  FOUND                Label


In [ ]:
# ============================================================
# WTMC2021 — LOAD FROZEN-PROTOCOL COLUMNS + VERIFY MONDAY
# ============================================================

import os
import zipfile
import numpy as np
import pandas as pd

ZIP_PATH = "/content/wtmc2021_dataset.zip"

if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(
        f"Dataset ZIP not found: {ZIP_PATH}"
    )

USE_COLUMNS = [
    "Timestamp",
    "Src IP",
    "Dst IP",
    "Flow IAT Max",
    "Bwd Segment Size Avg",
    "Down/Up Ratio",
    "Label",
]

DAY_ORDER = {
    "Monday": 1,
    "Tuesday": 2,
    "Wednesday": 3,
    "Thursday": 4,
    "Friday": 5,
}

frames = []

with zipfile.ZipFile(ZIP_PATH) as archive:

    csv_files = [
        name
        for name in archive.namelist()
        if name.lower().endswith(".csv")
        and not name.endswith("/")
    ]

    csv_files = sorted(
        csv_files,
        key=lambda name: next(
            (
                order
                for day, order in DAY_ORDER.items()
                if day.lower() in name.lower()
            ),
            999,
        ),
    )

    print("Files to load:")
    for filename in csv_files:
        print(" ", filename)

    for filename in csv_files:

        day_name = next(
            (
                day
                for day in DAY_ORDER
                if day.lower() in filename.lower()
            ),
            "Unknown",
        )

        print(f"\nLoading {filename} ...")

        with archive.open(filename) as stream:
            current = pd.read_csv(
                stream,
                usecols=USE_COLUMNS,
                low_memory=False,
            )

        current["_dataset_day"] = day_name
        current["_source_file"] = filename

        frames.append(current)

        print(
            f"  Rows: {len(current):,}"
        )

wtmc = pd.concat(
    frames,
    ignore_index=True,
)

del frames

print("\n" + "=" * 100)
print("RAW COMBINED DATA")
print("=" * 100)

print(f"Rows:    {len(wtmc):,}")
print(f"Columns: {len(wtmc.columns):,}")

# ============================================================
# 1. NORMALIZE IDENTIFIERS AND LABELS
# ============================================================

wtmc["Src IP"] = (
    wtmc["Src IP"]
    .astype(str)
    .str.strip()
)

wtmc["Dst IP"] = (
    wtmc["Dst IP"]
    .astype(str)
    .str.strip()
)

wtmc["Label"] = (
    wtmc["Label"]
    .astype(str)
    .str.strip()
)

wtmc["_label_normalized"] = (
    wtmc["Label"]
    .str.upper()
    .str.strip()
)

wtmc["_is_attack"] = (
    wtmc["_label_normalized"]
    .ne("BENIGN")
)

# ============================================================
# 2. PARSE TIMESTAMP
# ============================================================

wtmc["Timestamp"] = pd.to_datetime(
    wtmc["Timestamp"],
    errors="coerce",
    dayfirst=True,
    format="mixed",
)

timestamp_failures = int(
    wtmc["Timestamp"].isna().sum()
)

print("\nTimestamp parsing:")
print(
    f"  Valid:   "
    f"{wtmc['Timestamp'].notna().sum():,}"
)
print(
    f"  Failed:  "
    f"{timestamp_failures:,}"
)
print(
    f"  Failure rate: "
    f"{timestamp_failures / len(wtmc):.6%}"
)

# ============================================================
# 3. NUMERIC CONVERSION
# ============================================================

NUMERIC_COLUMNS = [
    "Flow IAT Max",
    "Bwd Segment Size Avg",
    "Down/Up Ratio",
]

for column in NUMERIC_COLUMNS:
    wtmc[column] = (
        pd.to_numeric(
            wtmc[column],
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
    )

# ============================================================
# 4. REMOVE INVALID CORE ROWS
# ============================================================

valid_core_mask = (
    wtmc["Timestamp"].notna()
    &
    wtmc["Src IP"].ne("")
    &
    wtmc["Dst IP"].ne("")
    &
    wtmc["Src IP"].str.lower().ne("nan")
    &
    wtmc["Dst IP"].str.lower().ne("nan")
)

invalid_core_rows = int(
    (~valid_core_mask).sum()
)

wtmc = (
    wtmc.loc[
        valid_core_mask
    ]
    .reset_index(drop=True)
)

print("\nCore validation:")
print(
    f"  Invalid rows removed: "
    f"{invalid_core_rows:,}"
)
print(
    f"  Remaining rows:       "
    f"{len(wtmc):,}"
)

# ============================================================
# 5. PER-DAY SUMMARY
# ============================================================

day_summary = (
    wtmc
    .groupby(
        "_dataset_day",
        observed=True,
    )
    .agg(
        Flows=(
            "Label",
            "size",
        ),
        Attack_Flows=(
            "_is_attack",
            "sum",
        ),
        Unique_Sources=(
            "Src IP",
            "nunique",
        ),
        Unique_Destinations=(
            "Dst IP",
            "nunique",
        ),
        Start_Time=(
            "Timestamp",
            "min",
        ),
        End_Time=(
            "Timestamp",
            "max",
        ),
    )
    .reset_index()
)

day_summary["Attack_Rate"] = (
    day_summary["Attack_Flows"]
    /
    day_summary["Flows"]
)

day_summary["_day_order"] = (
    day_summary["_dataset_day"]
    .map(DAY_ORDER)
)

day_summary = (
    day_summary
    .sort_values("_day_order")
    .drop(columns="_day_order")
)

print("\n" + "=" * 100)
print("PER-DAY SUMMARY")
print("=" * 100)

print(
    day_summary.to_string(
        index=False,
        formatters={
            "Attack_Rate": "{:.6%}".format,
        },
    )
)

# ============================================================
# 6. MONDAY BENIGN-BASELINE CHECK
# ============================================================

monday = (
    wtmc[
        wtmc["_dataset_day"]
        .eq("Monday")
    ]
    .copy()
)

monday_attack_count = int(
    monday["_is_attack"].sum()
)

monday_labels = (
    monday["Label"]
    .value_counts(dropna=False)
)

print("\n" + "=" * 100)
print("MONDAY BASELINE VERIFICATION")
print("=" * 100)

print(f"Monday flows:        {len(monday):,}")
print(f"Monday attack flows: {monday_attack_count:,}")

print("\nMonday labels:")
print(monday_labels.to_string())

if monday_attack_count == 0:
    print(
        "\nPASS: Monday is a fully benign temporal calibration baseline."
    )
else:
    print(
        "\nFAIL: Monday contains attack-labeled flows."
    )
    print(
        "Do not proceed with frozen calibration until these rows are examined."
    )

# ============================================================
# 7. FEATURE COMPLETENESS
# ============================================================

feature_completeness = pd.DataFrame({
    "Feature": NUMERIC_COLUMNS,
    "Missing_Count": [
        int(wtmc[column].isna().sum())
        for column in NUMERIC_COLUMNS
    ],
    "Missing_Rate": [
        float(wtmc[column].isna().mean())
        for column in NUMERIC_COLUMNS
    ],
})

print("\n" + "=" * 100)
print("FEATURE COMPLETENESS")
print("=" * 100)

print(
    feature_completeness.to_string(
        index=False,
        formatters={
            "Missing_Rate": "{:.6%}".format,
        },
    )
)

# ============================================================
# 8. CREATE FROZEN SPLITS
# ============================================================

wtmc_calibration = (
    wtmc[
        wtmc["_dataset_day"]
        .eq("Monday")
    ]
    .copy()
    .reset_index(drop=True)
)

wtmc_evaluation = (
    wtmc[
        ~wtmc["_dataset_day"]
        .eq("Monday")
    ]
    .copy()
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("FROZEN TEMPORAL SPLIT")
print("=" * 100)

print(
    f"Calibration — Monday:       "
    f"{len(wtmc_calibration):,} flows"
)
print(
    f"Evaluation — Tue to Friday: "
    f"{len(wtmc_evaluation):,} flows"
)

print("\nObjects created:")
print("  wtmc")
print("  wtmc_calibration")
print("  wtmc_evaluation")
print("  day_summary")
print("  feature_completeness")

Files to load:
  Monday-WorkingHours.csv
  Tuesday-WorkingHours.csv
  Wednesday-WorkingHours.csv
  Thursday-WorkingHours.csv
  Friday-WorkingHours.csv

Loading Monday-WorkingHours.csv ...
  Rows: 371,749

Loading Tuesday-WorkingHours.csv ...
  Rows: 322,003

Loading Wednesday-WorkingHours.csv ...
  Rows: 496,779

Loading Thursday-WorkingHours.csv ...
  Rows: 362,368

Loading Friday-WorkingHours.csv ...
  Rows: 547,915

RAW COMBINED DATA
Rows:    2,100,814
Columns: 9

Timestamp parsing:
  Valid:   2,100,814
  Failed:  0
  Failure rate: 0.000000%

Core validation:
  Invalid rows removed: 0
  Remaining rows:       2,100,814

PER-DAY SUMMARY
_dataset_day  Flows  Attack_Flows  Unique_Sources  Unique_Destinations          Start_Time            End_Time Attack_Rate
      Monday 371749             0              30                 9698 2017-07-03 13:55:58 2017-07-03 22:01:34   0.000000%
     Tuesday 322003          6972              22                 8490 2017-07-04 13:53:44 2017-07-04 22:00:

In [ ]:
# ============================================================
# WTMC2021 — FROZEN-PROTOCOL VALIDATION
#
# Frozen specification developed on CIC-IDS2017:
#
# Window:
#   1 hour
#
# Structural:
#   S_BV =
#       active-span flow rate >= Monday P99
#       AND flow count >= Monday P99
#
#   S_REP =
#       flow_count / unique_dst_ip >= Monday P99
#       AND flow_count >= Monday P50
#
#   S_T = S_BV OR S_REP
#
# Behavioral flow-level anomaly:
#   two-sided Monday P01/P99 empirical tails
#
#   B_PACKET:
#       Bwd Segment Size Avg
#       window outlier count >= Monday P95
#       AND window outlier share >= Monday P95
#
#   B_TEMPORAL:
#       Flow IAT Max
#       window outlier count >= Monday P95
#
#   B_DIRECTIONAL:
#       Down/Up Ratio
#       window outlier count >= Monday P95
#
#   B_MULTI =
#       B_PACKET OR B_TEMPORAL OR B_DIRECTIONAL
#
# Hybrid:
#   H_MULTI = S_T AND B_MULTI
#
# Labels are used only for post-hoc evaluation.
# ============================================================

import math
import numpy as np
import pandas as pd

try:
    from scipy.stats import hypergeom
    SCIPY_AVAILABLE = True
except ImportError:
    SCIPY_AVAILABLE = False

print("=" * 125)
print("WTMC2021 — FROZEN-PROTOCOL VALIDATION")
print("=" * 125)

# ============================================================
# 1. VERIFY INPUT OBJECTS
# ============================================================

required_objects = [
    "wtmc",
    "wtmc_calibration",
    "wtmc_evaluation",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise NameError(
        f"Missing objects: {missing_objects}\n"
        "Run the WTMC2021 loading and Monday-verification cell first."
    )

required_columns = [
    "Timestamp",
    "Src IP",
    "Dst IP",
    "Flow IAT Max",
    "Bwd Segment Size Avg",
    "Down/Up Ratio",
    "Label",
    "_dataset_day",
    "_is_attack",
]

missing_columns = [
    column
    for column in required_columns
    if column not in wtmc.columns
]

if missing_columns:
    raise KeyError(
        f"WTMC2021 table is missing: {missing_columns}"
    )

# ============================================================
# 2. FROZEN CONFIGURATION
# ============================================================

WINDOW_FREQUENCY = "1h"

STRUCTURAL_RATE_PERCENTILE = 0.99
STRUCTURAL_COUNT_PERCENTILE = 0.99
REPETITION_PERCENTILE = 0.99
REPETITION_MIN_COUNT_PERCENTILE = 0.50

BEHAVIORAL_FLOW_TAIL_PERCENTILE = 0.99
BEHAVIORAL_WINDOW_PERCENTILE = 0.95

KEY_COLUMNS = [
    "_src",
    "_window_start",
]

BEHAVIORAL_FEATURES = {
    "PACKET": "Bwd Segment Size Avg",
    "TEMPORAL": "Flow IAT Max",
    "DIRECTIONAL": "Down/Up Ratio",
}

print("\nFrozen protocol:")
print("  Window: 1 hour")
print("  S_BV:   rate P99 AND count P99")
print("  S_REP:  repetition P99 AND count P50")
print("  Packet: JOINT window P95")
print("  Temporal: COUNT window P95")
print("  Directional: COUNT window P95")
print("  No feature or percentile tuning is performed.")

# ============================================================
# 3. CANONICAL FLOW TABLE
# ============================================================

flows = (
    wtmc[
        [
            "Timestamp",
            "Src IP",
            "Dst IP",
            "Flow IAT Max",
            "Bwd Segment Size Avg",
            "Down/Up Ratio",
            "Label",
            "_dataset_day",
            "_is_attack",
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

flows = flows.rename(
    columns={
        "Timestamp": "_timestamp",
        "Src IP": "_src",
        "Dst IP": "_dst",
        "Label": "_label",
    }
)

flows["_src"] = (
    flows["_src"]
    .astype(str)
    .str.strip()
)

flows["_dst"] = (
    flows["_dst"]
    .astype(str)
    .str.strip()
)

flows["_label"] = (
    flows["_label"]
    .astype(str)
    .str.strip()
)

flows["_timestamp"] = pd.to_datetime(
    flows["_timestamp"],
    errors="coerce",
)

for feature in BEHAVIORAL_FEATURES.values():
    flows[feature] = (
        pd.to_numeric(
            flows[feature],
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
    )

flows["_window_start"] = (
    flows["_timestamp"]
    .dt.floor(WINDOW_FREQUENCY)
)

flows["weekday"] = (
    flows["_window_start"]
    .dt.day_name()
)

flows["_is_attack"] = (
    flows["_is_attack"]
    .fillna(False)
    .astype(bool)
)

invalid_core = (
    flows["_timestamp"].isna()
    |
    flows["_src"].eq("")
    |
    flows["_dst"].eq("")
    |
    flows["_src"].str.lower().eq("nan")
    |
    flows["_dst"].str.lower().eq("nan")
)

if invalid_core.any():
    raise ValueError(
        f"Invalid core flow rows detected: "
        f"{int(invalid_core.sum()):,}"
    )

print("\nCanonical flow table:")
print(f"  Rows: {len(flows):,}")
print(
    f"  Calibration flows: "
    f"{flows['_dataset_day'].eq('Monday').sum():,}"
)
print(
    f"  Evaluation flows:  "
    f"{(~flows['_dataset_day'].eq('Monday')).sum():,}"
)

# ============================================================
# 4. VERIFY FROZEN TEMPORAL SPLIT
# ============================================================

monday_flow_mask = (
    flows["_dataset_day"]
    .eq("Monday")
)

evaluation_flow_mask = (
    ~monday_flow_mask
)

monday_attack_flows = int(
    flows.loc[
        monday_flow_mask,
        "_is_attack",
    ].sum()
)

if monday_attack_flows != 0:
    raise ValueError(
        "Monday contains attack-labeled flows. "
        "Frozen benign calibration cannot proceed."
    )

print(
    "\nMonday benign-calibration check: PASS "
    f"({monday_attack_flows} attack flows)"
)

# ============================================================
# 5. BUILD SOURCE-WINDOW STRUCTURAL FEATURES
# ============================================================

source_windows = (
    flows
    .groupby(
        KEY_COLUMNS,
        observed=True,
        sort=False,
    )
    .agg(
        dataset_day=(
            "_dataset_day",
            "first",
        ),

        flow_count=(
            "_dst",
            "size",
        ),

        unique_dst_ip=(
            "_dst",
            "nunique",
        ),

        first_flow_time=(
            "_timestamp",
            "min",
        ),

        last_flow_time=(
            "_timestamp",
            "max",
        ),

        attack_flow_count=(
            "_is_attack",
            "sum",
        ),
    )
    .reset_index()
)

source_windows[
    "activity_duration_seconds"
] = (
    source_windows[
        "last_flow_time"
    ]
    -
    source_windows[
        "first_flow_time"
    ]
).dt.total_seconds()

effective_duration = (
    source_windows[
        "activity_duration_seconds"
    ]
    .clip(lower=1.0)
)

source_windows[
    "active_span_flow_rate"
] = (
    source_windows[
        "flow_count"
    ]
    /
    effective_duration
)

source_windows[
    "destination_repetition"
] = (
    source_windows[
        "flow_count"
    ]
    /
    source_windows[
        "unique_dst_ip"
    ].replace(
        0,
        np.nan,
    )
).replace(
    [np.inf, -np.inf],
    np.nan,
).fillna(0)

source_windows[
    "attack_flow_share"
] = (
    source_windows[
        "attack_flow_count"
    ]
    /
    source_windows[
        "flow_count"
    ].replace(
        0,
        np.nan,
    )
).fillna(0)

source_windows["weekday"] = (
    source_windows[
        "_window_start"
    ].dt.day_name()
)

# ============================================================
# 6. DOMINANT ATTACK LABEL PER WINDOW
# ============================================================

attack_label_counts = (
    flows[
        flows["_is_attack"]
    ]
    .groupby(
        [
            "_src",
            "_window_start",
            "_label",
        ],
        observed=True,
    )
    .size()
    .rename(
        "dominant_attack_label_flows"
    )
    .reset_index()
)

dominant_attack_labels = (
    attack_label_counts
    .sort_values(
        [
            "_src",
            "_window_start",
            "dominant_attack_label_flows",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
    .groupby(
        KEY_COLUMNS,
        as_index=False,
    )
    .first()
    .rename(
        columns={
            "_label": (
                "dominant_attack_label"
            )
        }
    )
)

source_windows = (
    source_windows
    .merge(
        dominant_attack_labels[
            [
                "_src",
                "_window_start",
                "dominant_attack_label",
                "dominant_attack_label_flows",
            ]
        ],
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    .reset_index(drop=True)
)

structural_calibration = (
    source_windows[
        source_windows[
            "dataset_day"
        ].eq("Monday")
    ]
    .copy()
    .reset_index(drop=True)
)

structural_evaluation = (
    source_windows[
        ~source_windows[
            "dataset_day"
        ].eq("Monday")
    ]
    .copy()
    .reset_index(drop=True)
)

print("\nSource-window populations:")
print(
    f"  Monday calibration windows: "
    f"{len(structural_calibration):,}"
)
print(
    f"  Evaluation windows:         "
    f"{len(structural_evaluation):,}"
)
print(
    f"  Any-attack windows:         "
    f"{(structural_evaluation['attack_flow_count'] > 0).sum():,}"
)
print(
    f"  Attack-dominant windows:    "
    f"{(structural_evaluation['attack_flow_share'] >= 0.50).sum():,}"
)

# ============================================================
# 7. MONDAY STRUCTURAL THRESHOLDS
# ============================================================

threshold_rate_p99 = (
    structural_calibration[
        "active_span_flow_rate"
    ]
    .quantile(
        STRUCTURAL_RATE_PERCENTILE
    )
)

threshold_count_p99 = (
    structural_calibration[
        "flow_count"
    ]
    .quantile(
        STRUCTURAL_COUNT_PERCENTILE
    )
)

threshold_repetition_p99 = (
    structural_calibration[
        "destination_repetition"
    ]
    .quantile(
        REPETITION_PERCENTILE
    )
)

threshold_count_p50 = (
    structural_calibration[
        "flow_count"
    ]
    .quantile(
        REPETITION_MIN_COUNT_PERCENTILE
    )
)

structural_evaluation["S_BV"] = (
    (
        structural_evaluation[
            "active_span_flow_rate"
        ]
        >= threshold_rate_p99
    )
    &
    (
        structural_evaluation[
            "flow_count"
        ]
        >= threshold_count_p99
    )
)

structural_evaluation["S_REP"] = (
    (
        structural_evaluation[
            "destination_repetition"
        ]
        >= threshold_repetition_p99
    )
    &
    (
        structural_evaluation[
            "flow_count"
        ]
        >= threshold_count_p50
    )
)

structural_evaluation["S_T"] = (
    structural_evaluation["S_BV"]
    |
    structural_evaluation["S_REP"]
)

# ============================================================
# 8. FLOW-LEVEL BEHAVIORAL TAILS
# ============================================================

behavioral_flow_threshold_rows = []

for mechanism, feature in (
    BEHAVIORAL_FEATURES.items()
):
    monday_values = (
        flows.loc[
            monday_flow_mask,
            feature,
        ]
        .dropna()
    )

    lower_threshold = (
        monday_values.quantile(
            1.0
            -
            BEHAVIORAL_FLOW_TAIL_PERCENTILE
        )
    )

    upper_threshold = (
        monday_values.quantile(
            BEHAVIORAL_FLOW_TAIL_PERCENTILE
        )
    )

    outlier_column = (
        f"_outlier_{mechanism.lower()}"
    )

    valid_column = (
        f"_valid_{mechanism.lower()}"
    )

    flows[valid_column] = (
        flows[feature].notna()
    )

    flows[outlier_column] = (
        flows[feature]
        .lt(lower_threshold)
        |
        flows[feature]
        .gt(upper_threshold)
    ).fillna(False)

    behavioral_flow_threshold_rows.append({
        "Mechanism": mechanism,
        "Feature": feature,
        "Monday_Lower_P01": (
            lower_threshold
        ),
        "Monday_Upper_P99": (
            upper_threshold
        ),
        "Monday_Valid_Flows": int(
            monday_values.size
        ),
        "Monday_Flow_Outlier_Rate": float(
            flows.loc[
                monday_flow_mask,
                outlier_column,
            ].mean()
        ),
    })

behavioral_flow_thresholds = pd.DataFrame(
    behavioral_flow_threshold_rows
)

# ============================================================
# 9. AGGREGATE BEHAVIORAL EVIDENCE TO WINDOWS
# ============================================================

aggregation_specification = {}

for mechanism in BEHAVIORAL_FEATURES:
    mechanism_lower = (
        mechanism.lower()
    )

    aggregation_specification[
        f"{mechanism_lower}_valid_count"
    ] = (
        f"_valid_{mechanism_lower}",
        "sum",
    )

    aggregation_specification[
        f"{mechanism_lower}_outlier_count"
    ] = (
        f"_outlier_{mechanism_lower}",
        "sum",
    )

behavioral_windows = (
    flows
    .groupby(
        KEY_COLUMNS,
        observed=True,
        sort=False,
    )
    .agg(
        dataset_day=(
            "_dataset_day",
            "first",
        ),

        **aggregation_specification,
    )
    .reset_index()
)

for mechanism in BEHAVIORAL_FEATURES:
    mechanism_lower = (
        mechanism.lower()
    )

    behavioral_windows[
        f"{mechanism_lower}_outlier_share"
    ] = (
        behavioral_windows[
            f"{mechanism_lower}_outlier_count"
        ]
        /
        behavioral_windows[
            f"{mechanism_lower}_valid_count"
        ].replace(
            0,
            np.nan,
        )
    ).fillna(0)

behavioral_calibration = (
    behavioral_windows[
        behavioral_windows[
            "dataset_day"
        ].eq("Monday")
    ]
    .copy()
    .reset_index(drop=True)
)

behavioral_evaluation = (
    behavioral_windows[
        ~behavioral_windows[
            "dataset_day"
        ].eq("Monday")
    ]
    .copy()
    .reset_index(drop=True)
)

# ============================================================
# 10. MONDAY WINDOW-LEVEL BEHAVIORAL THRESHOLDS
# ============================================================

behavioral_window_thresholds = {}

for mechanism in BEHAVIORAL_FEATURES:
    mechanism_lower = (
        mechanism.lower()
    )

    count_column = (
        f"{mechanism_lower}_outlier_count"
    )

    share_column = (
        f"{mechanism_lower}_outlier_share"
    )

    count_threshold = (
        behavioral_calibration[
            count_column
        ]
        .quantile(
            BEHAVIORAL_WINDOW_PERCENTILE
        )
    )

    share_threshold = (
        behavioral_calibration[
            share_column
        ]
        .quantile(
            BEHAVIORAL_WINDOW_PERCENTILE
        )
    )

    behavioral_window_thresholds[
        mechanism
    ] = {
        "count": count_threshold,
        "share": share_threshold,
    }

behavioral_window_threshold_table = pd.DataFrame(
    [
        {
            "Mechanism": mechanism,
            "Feature": (
                BEHAVIORAL_FEATURES[
                    mechanism
                ]
            ),
            "Monday_Count_P95": (
                thresholds["count"]
            ),
            "Monday_Share_P95": (
                thresholds["share"]
            ),
        }
        for mechanism, thresholds
        in behavioral_window_thresholds.items()
    ]
)

# ============================================================
# 11. APPLY FROZEN BEHAVIORAL RULES
# ============================================================

def count_rule(
    frame,
    column,
    threshold,
):
    if threshold <= 0:
        return (
            frame[column]
            > 0
        )

    return (
        frame[column]
        >= threshold
    )


def share_rule(
    frame,
    column,
    threshold,
):
    if threshold <= 0:
        return (
            frame[column]
            > 0
        )

    return (
        frame[column]
        >= threshold
    )


# Packet: count AND share P95
packet_count_condition = count_rule(
    behavioral_evaluation,
    "packet_outlier_count",
    behavioral_window_thresholds[
        "PACKET"
    ]["count"],
)

packet_share_condition = share_rule(
    behavioral_evaluation,
    "packet_outlier_share",
    behavioral_window_thresholds[
        "PACKET"
    ]["share"],
)

behavioral_evaluation["B_PACKET"] = (
    packet_count_condition
    &
    packet_share_condition
)

# Temporal: count P95
behavioral_evaluation["B_TEMPORAL"] = (
    count_rule(
        behavioral_evaluation,
        "temporal_outlier_count",
        behavioral_window_thresholds[
            "TEMPORAL"
        ]["count"],
    )
)

# Directional: count P95
behavioral_evaluation[
    "B_DIRECTIONAL"
] = (
    count_rule(
        behavioral_evaluation,
        "directional_outlier_count",
        behavioral_window_thresholds[
            "DIRECTIONAL"
        ]["count"],
    )
)

behavioral_evaluation["B_MULTI"] = (
    behavioral_evaluation[
        "B_PACKET"
    ]
    |
    behavioral_evaluation[
        "B_TEMPORAL"
    ]
    |
    behavioral_evaluation[
        "B_DIRECTIONAL"
    ]
)

behavioral_evaluation[
    "behavioral_mechanism_count"
] = (
    behavioral_evaluation[
        [
            "B_PACKET",
            "B_TEMPORAL",
            "B_DIRECTIONAL",
        ]
    ]
    .astype(int)
    .sum(axis=1)
)

# ============================================================
# 12. FUSE STRUCTURAL AND BEHAVIORAL TABLES
# ============================================================

behavioral_columns_for_merge = [
    "_src",
    "_window_start",
    "packet_valid_count",
    "packet_outlier_count",
    "packet_outlier_share",
    "temporal_valid_count",
    "temporal_outlier_count",
    "temporal_outlier_share",
    "directional_valid_count",
    "directional_outlier_count",
    "directional_outlier_share",
    "B_PACKET",
    "B_TEMPORAL",
    "B_DIRECTIONAL",
    "B_MULTI",
    "behavioral_mechanism_count",
]

frozen_validation = (
    structural_evaluation
    .merge(
        behavioral_evaluation[
            behavioral_columns_for_merge
        ],
        on=KEY_COLUMNS,
        how="left",
        validate="one_to_one",
    )
    .reset_index(drop=True)
)

for column in [
    "B_PACKET",
    "B_TEMPORAL",
    "B_DIRECTIONAL",
    "B_MULTI",
]:
    frozen_validation[column] = (
        frozen_validation[column]
        .fillna(False)
        .astype(bool)
    )

frozen_validation[
    "behavioral_mechanism_count"
] = (
    frozen_validation[
        "behavioral_mechanism_count"
    ]
    .fillna(0)
    .astype(int)
)

frozen_validation["H_MULTI"] = (
    frozen_validation["S_T"]
    &
    frozen_validation["B_MULTI"]
)

# ============================================================
# 13. POSITIONAL METRIC FUNCTIONS
# ============================================================

def as_bool_array(values):
    return np.asarray(
        values,
        dtype=bool,
    )


def evaluate_prediction(
    prediction,
    truth,
    frame,
):
    prediction = as_bool_array(
        prediction
    )

    truth = as_bool_array(
        truth
    )

    if (
        len(prediction) != len(truth)
        or len(prediction) != len(frame)
    ):
        raise ValueError(
            "Metric inputs have different lengths."
        )

    tp_mask = (
        prediction
        &
        truth
    )

    fp_mask = (
        prediction
        &
        ~truth
    )

    fn_mask = (
        ~prediction
        &
        truth
    )

    tn_mask = (
        ~prediction
        &
        ~truth
    )

    tp = int(tp_mask.sum())
    fp = int(fp_mask.sum())
    fn = int(fn_mask.sum())
    tn = int(tn_mask.sum())

    flagged = int(
        prediction.sum()
    )

    precision = (
        tp / (tp + fp)
        if (tp + fp) > 0
        else 0.0
    )

    recall = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0.0
    )

    base_rate = float(
        truth.mean()
    )

    lift = (
        precision / base_rate
        if base_rate > 0
        else np.nan
    )

    return {
        "Flagged": flagged,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "FPR": fpr,
        "Base_Rate": base_rate,
        "Lift": lift,

        "Unique_Flagged_Sources": (
            frame.loc[
                prediction,
                "_src",
            ].nunique()
        ),

        "Unique_TP_Sources": (
            frame.loc[
                tp_mask,
                "_src",
            ].nunique()
        ),

        "Mean_Attack_Share_Flagged": (
            frame.loc[
                prediction,
                "attack_flow_share",
            ].mean()
            if flagged > 0
            else np.nan
        ),

        "Median_Attack_Share_Flagged": (
            frame.loc[
                prediction,
                "attack_flow_share",
            ].median()
            if flagged > 0
            else np.nan
        ),
    }


def calculate_rci(
    structural_prediction,
    behavioral_prediction,
):
    S = as_bool_array(
        structural_prediction
    )

    B = as_bool_array(
        behavioral_prediction
    )

    n = len(S)

    s_size = int(S.sum())
    b_size = int(B.sum())

    intersection = int(
        (
            S
            &
            B
        ).sum()
    )

    union = int(
        (
            S
            |
            B
        ).sum()
    )

    observed_rci = (
        intersection / union
        if union > 0
        else 0.0
    )

    s_rate = (
        s_size / n
        if n > 0
        else 0.0
    )

    b_rate = (
        b_size / n
        if n > 0
        else 0.0
    )

    expected_intersection_rate = (
        s_rate * b_rate
    )

    expected_union_rate = (
        s_rate
        +
        b_rate
        -
        expected_intersection_rate
    )

    expected_rci = (
        expected_intersection_rate
        /
        expected_union_rate
        if expected_union_rate > 0
        else 0.0
    )

    overlap_p = np.nan
    overlap_log10_p = np.nan

    if SCIPY_AVAILABLE:
        overlap_p = float(
            hypergeom.sf(
                intersection - 1,
                n,
                s_size,
                b_size,
            )
        )

        log_sf = float(
            hypergeom.logsf(
                intersection - 1,
                n,
                s_size,
                b_size,
            )
        )

        if np.isfinite(log_sf):
            overlap_log10_p = (
                log_sf
                /
                math.log(10)
            )

    return {
        "Evaluation_Units": n,
        "S_T_Size": s_size,
        "B_MULTI_Size": b_size,
        "Intersection_Size": intersection,
        "Union_Size": union,
        "RCI_Jaccard": observed_rci,
        "Expected_RCI": expected_rci,

        "RCI_Lift": (
            observed_rci
            /
            expected_rci
            if expected_rci > 0
            else np.nan
        ),

        "Coverage_S_to_B": (
            intersection / s_size
            if s_size > 0
            else 0.0
        ),

        "Coverage_B_to_S": (
            intersection / b_size
            if b_size > 0
            else 0.0
        ),

        "Hypergeometric_Overlap_P": (
            overlap_p
        ),

        "Hypergeometric_Log10_P": (
            overlap_log10_p
        ),

        "S_Subset_B": bool(
            (
                S
                &
                ~B
            ).sum() == 0
        ),

        "B_Subset_S": bool(
            (
                B
                &
                ~S
            ).sum() == 0
        ),
    }

# ============================================================
# 14. EVALUATE FROZEN MODELS
# ============================================================

truth_definitions = {
    "Any_Attack_Flow": (
        frozen_validation[
            "attack_flow_count"
        ]
        > 0
    ),

    "AttackShare_GE_1pct": (
        frozen_validation[
            "attack_flow_share"
        ]
        >= 0.01
    ),

    "AttackShare_GE_50pct": (
        frozen_validation[
            "attack_flow_share"
        ]
        >= 0.50
    ),

    "AttackShare_GE_99pct": (
        frozen_validation[
            "attack_flow_share"
        ]
        >= 0.99
    ),
}

model_definitions = {
    "S_BV": (
        frozen_validation["S_BV"]
    ),

    "S_REP": (
        frozen_validation["S_REP"]
    ),

    "S_T": (
        frozen_validation["S_T"]
    ),

    "B_PACKET": (
        frozen_validation["B_PACKET"]
    ),

    "B_TEMPORAL": (
        frozen_validation["B_TEMPORAL"]
    ),

    "B_DIRECTIONAL": (
        frozen_validation[
            "B_DIRECTIONAL"
        ]
    ),

    "B_MULTI": (
        frozen_validation["B_MULTI"]
    ),

    "H_MULTI": (
        frozen_validation["H_MULTI"]
    ),
}

metric_rows = []

for truth_name, truth in (
    truth_definitions.items()
):
    for model_name, prediction in (
        model_definitions.items()
    ):
        metrics = evaluate_prediction(
            prediction,
            truth,
            frozen_validation,
        )

        metric_rows.append({
            "Model": model_name,
            "Truth_Definition": truth_name,
            **metrics,
        })

frozen_model_evaluation = pd.DataFrame(
    metric_rows
)

# ============================================================
# 15. RCI
# ============================================================

frozen_rci = pd.DataFrame(
    [
        calculate_rci(
            frozen_validation["S_T"],
            frozen_validation["B_MULTI"],
        )
    ]
)

# ============================================================
# 16. FAMILY COVERAGE
# ============================================================

family_rows = []

dominant_windows = (
    frozen_validation[
        frozen_validation[
            "attack_flow_share"
        ]
        >= 0.50
    ]
    .copy()
)

for model_name in [
    "S_T",
    "B_MULTI",
    "H_MULTI",
]:
    dominant_windows[
        "_detected"
    ] = (
        dominant_windows[
            model_name
        ]
        .astype(bool)
    )

    for family_name, family_group in (
        dominant_windows
        .dropna(
            subset=[
                "dominant_attack_label"
            ]
        )
        .groupby(
            "dominant_attack_label",
            observed=True,
        )
    ):
        detected = (
            family_group[
                family_group[
                    "_detected"
                ]
            ]
        )

        family_rows.append({
            "Model": model_name,
            "Attack_Family": (
                family_name
            ),
            "Attack_Dominant_Windows": (
                len(family_group)
            ),
            "Detected_Windows": (
                len(detected)
            ),
            "Family_Recall": (
                len(detected)
                /
                len(family_group)
                if len(family_group) > 0
                else np.nan
            ),
            "Unique_Attack_Sources": (
                family_group[
                    "_src"
                ].nunique()
            ),
            "Detected_Sources": (
                detected[
                    "_src"
                ].nunique()
            ),
            "Total_Attack_Flows": int(
                family_group[
                    "attack_flow_count"
                ].sum()
            ),
        })

frozen_family_coverage = pd.DataFrame(
    family_rows
)

# ============================================================
# 17. STRUCTURAL ALERT ATTRIBUTION
# ============================================================

frozen_structural_alerts = (
    frozen_validation[
        frozen_validation[
            "S_T"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)

frozen_structural_alerts[
    "attack_dominant"
] = (
    frozen_structural_alerts[
        "attack_flow_share"
    ]
    >= 0.50
)

# ============================================================
# 18. CONSISTENCY CHECKS
# ============================================================

dominant_truth = (
    frozen_validation[
        "attack_flow_share"
    ]
    .ge(0.50)
    .to_numpy(
        dtype=bool
    )
)

h_prediction = (
    frozen_validation[
        "H_MULTI"
    ]
    .to_numpy(
        dtype=bool
    )
)

direct_tp = int(
    (
        h_prediction
        &
        dominant_truth
    ).sum()
)

direct_fp = int(
    (
        h_prediction
        &
        ~dominant_truth
    ).sum()
)

reported_h = (
    frozen_model_evaluation[
        (
            frozen_model_evaluation[
                "Model"
            ].eq("H_MULTI")
        )
        &
        (
            frozen_model_evaluation[
                "Truth_Definition"
            ].eq(
                "AttackShare_GE_50pct"
            )
        )
    ]
    .iloc[0]
)

assert (
    direct_tp
    ==
    int(reported_h["TP"])
), "H_MULTI TP consistency failed."

assert (
    direct_fp
    ==
    int(reported_h["FP"])
), "H_MULTI FP consistency failed."

print("\nConsistency checks: PASSED")

# ============================================================
# 19. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 125)
print("FROZEN STRUCTURAL THRESHOLDS")
print("=" * 125)

structural_threshold_table = pd.DataFrame(
    [
        {
            "Window": "1h",
            "BV_Rate_P99": (
                threshold_rate_p99
            ),
            "BV_Count_P99": (
                threshold_count_p99
            ),
            "REP_Ratio_P99": (
                threshold_repetition_p99
            ),
            "REP_Count_P50": (
                threshold_count_p50
            ),
        }
    ]
)

print(
    structural_threshold_table
    .round(8)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print("FROZEN BEHAVIORAL FLOW THRESHOLDS")
print("=" * 125)

print(
    behavioral_flow_thresholds
    .round(8)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print("FROZEN BEHAVIORAL WINDOW THRESHOLDS")
print("=" * 125)

print(
    behavioral_window_threshold_table
    .round(8)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print(
    "FROZEN-PROTOCOL PERFORMANCE "
    "— ATTACK SHARE >= 50%"
)
print("=" * 125)

primary_results = (
    frozen_model_evaluation[
        frozen_model_evaluation[
            "Truth_Definition"
        ].eq(
            "AttackShare_GE_50pct"
        )
    ]
    .sort_values(
        [
            "F1",
            "Precision",
            "Recall",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
)

print(
    primary_results
    .round(8)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print("FROZEN TEMPORAL RCI")
print("=" * 125)

print(
    frozen_rci
    .round(12)
    .to_string(index=False)
)

print("\n" + "=" * 125)
print("FROZEN STRUCTURAL ALERT ATTRIBUTION")
print("=" * 125)

alert_columns = [
    "_src",
    "_window_start",
    "dataset_day",
    "dominant_attack_label",
    "flow_count",
    "unique_dst_ip",
    "active_span_flow_rate",
    "destination_repetition",
    "attack_flow_count",
    "attack_flow_share",
    "S_BV",
    "S_REP",
    "S_T",
    "B_PACKET",
    "B_TEMPORAL",
    "B_DIRECTIONAL",
    "behavioral_mechanism_count",
    "B_MULTI",
    "H_MULTI",
    "attack_dominant",
]

print(
    frozen_structural_alerts[
        alert_columns
    ]
    .sort_values(
        [
            "attack_dominant",
            "attack_flow_share",
            "attack_flow_count",
        ],
        ascending=[
            False,
            False,
            False,
        ],
    )
    .to_string(
        index=False,
        formatters={
            "attack_flow_share": (
                "{:.2%}".format
            ),
        },
    )
)

print("\n" + "=" * 125)
print("FROZEN FAMILY COVERAGE — H_MULTI")
print("=" * 125)

print(
    frozen_family_coverage[
        frozen_family_coverage[
            "Model"
        ].eq("H_MULTI")
    ]
    .sort_values(
        [
            "Detected_Windows",
            "Family_Recall",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .to_string(
        index=False,
        formatters={
            "Family_Recall": (
                "{:.2%}".format
            ),
        },
    )
)

# ============================================================
# 20. SAVE OUTPUTS
# ============================================================

structural_threshold_table.to_csv(
    "/content/"
    "wtmc_frozen_structural_thresholds.csv",
    index=False,
)

behavioral_flow_thresholds.to_csv(
    "/content/"
    "wtmc_frozen_behavioral_flow_thresholds.csv",
    index=False,
)

behavioral_window_threshold_table.to_csv(
    "/content/"
    "wtmc_frozen_behavioral_window_thresholds.csv",
    index=False,
)

frozen_model_evaluation.to_csv(
    "/content/"
    "wtmc_frozen_model_evaluation.csv",
    index=False,
)

frozen_rci.to_csv(
    "/content/"
    "wtmc_frozen_RCI.csv",
    index=False,
)

frozen_family_coverage.to_csv(
    "/content/"
    "wtmc_frozen_family_coverage.csv",
    index=False,
)

frozen_structural_alerts.to_csv(
    "/content/"
    "wtmc_frozen_structural_alert_attribution.csv",
    index=False,
)

frozen_validation.to_csv(
    "/content/"
    "wtmc_frozen_full_source_windows.csv",
    index=False,
)

print("\n" + "=" * 125)
print("FILES SAVED")
print("=" * 125)

print(
    "/content/"
    "wtmc_frozen_structural_thresholds.csv"
)
print(
    "/content/"
    "wtmc_frozen_behavioral_flow_thresholds.csv"
)
print(
    "/content/"
    "wtmc_frozen_behavioral_window_thresholds.csv"
)
print(
    "/content/"
    "wtmc_frozen_model_evaluation.csv"
)
print(
    "/content/"
    "wtmc_frozen_RCI.csv"
)
print(
    "/content/"
    "wtmc_frozen_family_coverage.csv"
)
print(
    "/content/"
    "wtmc_frozen_structural_alert_attribution.csv"
)
print(
    "/content/"
    "wtmc_frozen_full_source_windows.csv"
)

WTMC2021 — FROZEN-PROTOCOL VALIDATION

Frozen protocol:
  Window: 1 hour
  S_BV:   rate P99 AND count P99
  S_REP:  repetition P99 AND count P50
  Packet: JOINT window P95
  Temporal: COUNT window P95
  Directional: COUNT window P95
  No feature or percentile tuning is performed.

Canonical flow table:
  Rows: 2,100,814
  Calibration flows: 371,749
  Evaluation flows:  1,729,065

Monday benign-calibration check: PASS (0 attack flows)

Source-window populations:
  Monday calibration windows: 143
  Evaluation windows:         593
  Any-attack windows:         35
  Attack-dominant windows:    16

Consistency checks: PASSED

FROZEN STRUCTURAL THRESHOLDS
Window  BV_Rate_P99  BV_Count_P99  REP_Ratio_P99  REP_Count_P50
    1h     5.611613      12075.94     715.509231         1540.0

FROZEN BEHAVIORAL FLOW THRESHOLDS
  Mechanism              Feature  Monday_Lower_P01  Monday_Upper_P99  Monday_Valid_Flows  Monday_Flow_Outlier_Rate
     PACKET Bwd Segment Size Avg               0.0      1.523642

## Structural ablation baselines

This section adds post-hoc ablation baselines **without changing the original frozen protocol**.

The purpose is to test whether the conjunctions used by the structural operators add measurable selectivity beyond their single-component threshold rules under the **same source-hour representation, calibration population, and ground-truth definition**.

Baselines:
- `COUNT_ONLY_P99`: flow count ≥ calibration P99
- `RATE_ONLY_P99`: active-span flow rate ≥ calibration P99
- `REP_ONLY_P99`: destination-repetition ratio ≥ calibration P99

Comparators:
- `S_BV = RATE_ONLY_P99 AND COUNT_ONLY_P99`
- `S_REP = REP_ONLY_P99 AND flow-count P50 guard`
- `S_T = S_BV OR S_REP`

The cell reports the same detection metrics used in the study and saves both the baseline table and direct conjunction-value diagnostics.


In [ ]:
# ============================================================
# STRUCTURAL ABLATION BASELINES — WTMC2021
#
# Purpose:
#   Test whether the SBV/SREP conjunctions add selectivity over
#   their single-component threshold rules under exactly the
#   same source-hour semantics, calibration population, and
#   AttackShare >= 50% ground-truth definition used in the paper.
#
# IMPORTANT:
#   This cell does NOT alter the frozen operators or any original
#   result. It adds post-hoc ablation baselines only.
# ============================================================

import numpy as np
import pandas as pd

required_objects = [
    "frozen_validation",
    "threshold_count_p99",
    "threshold_rate_p99",
    "threshold_repetition_p99",
    "evaluate_prediction",
]
missing = [name for name in required_objects if name not in globals()]
if missing:
    raise RuntimeError(
        "Run the original WTMC2021 validation cell first. "
        f"Missing objects: {missing}"
    )

ablation_predictions = {
    "COUNT_ONLY_P99": (
        frozen_validation["flow_count"] >= threshold_count_p99
    ),
    "RATE_ONLY_P99": (
        frozen_validation["active_span_flow_rate"] >= threshold_rate_p99
    ),
    "REP_ONLY_P99": (
        frozen_validation["destination_repetition"] >= threshold_repetition_p99
    ),
    "S_BV": frozen_validation["S_BV"],
    "S_REP": frozen_validation["S_REP"],
    "S_T": frozen_validation["S_T"],
}

truth_primary = (
    frozen_validation["attack_flow_share"] >= 0.50
)

ablation_rows = []
for model_name, prediction in ablation_predictions.items():
    metrics = evaluate_prediction(
        prediction,
        truth_primary,
        frozen_validation,
    )
    ablation_rows.append({
        "Environment": "WTMC2021",
        "Model": model_name,
        "Truth_Definition": "AttackShare_GE_50pct",
        **metrics,
    })

wtmc_structural_ablation = pd.DataFrame(ablation_rows)

preferred_columns = [
    "Environment",
    "Model",
    "Truth_Definition",
    "Flagged",
    "TP",
    "FP",
    "FN",
    "TN",
    "Precision",
    "Recall",
    "F1",
    "FPR",
    "Lift",
    "Unique_Flagged_Sources",
    "Unique_TP_Sources",
]
display_columns = [
    c for c in preferred_columns
    if c in wtmc_structural_ablation.columns
]

print("=" * 100)
print("WTMC2021 — STRUCTURAL ABLATION BASELINES")
print("=" * 100)
print(
    "Calibration thresholds reused unchanged:\n"
    f"  COUNT P99 = {threshold_count_p99:.6f}\n"
    f"  RATE  P99 = {threshold_rate_p99:.6f}\n"
    f"  REP   P99 = {threshold_repetition_p99:.6f}"
)
print()
print(
    wtmc_structural_ablation[display_columns]
    .to_string(index=False)
)

result_index = (
    wtmc_structural_ablation
    .set_index("Model")
)

diagnostic_pairs = [
    ("RATE_ONLY_P99", "S_BV", "SBV vs rate-only"),
    ("COUNT_ONLY_P99", "S_BV", "SBV vs count-only"),
    ("REP_ONLY_P99", "S_REP", "SREP vs repetition-only"),
]

diagnostic_rows = []
for baseline_name, operator_name, comparison in diagnostic_pairs:
    b = result_index.loc[baseline_name]
    o = result_index.loc[operator_name]

    diagnostic_rows.append({
        "Environment": "WTMC2021",
        "Comparison": comparison,
        "Baseline": baseline_name,
        "Operator": operator_name,
        "Delta_Flagged": int(o["Flagged"] - b["Flagged"]),
        "Delta_TP": int(o["TP"] - b["TP"]),
        "Delta_FP": int(o["FP"] - b["FP"]),
        "Delta_Precision": float(o["Precision"] - b["Precision"]),
        "Delta_Recall": float(o["Recall"] - b["Recall"]),
        "Delta_F1": float(o["F1"] - b["F1"]),
    })

wtmc_ablation_diagnostics = pd.DataFrame(diagnostic_rows)

print("\nConjunction-value diagnostics:")
print(
    wtmc_ablation_diagnostics
    .to_string(index=False)
)

wtmc_structural_ablation.to_csv(
    "/content/wtmc_structural_ablation_baselines.csv",
    index=False,
)
wtmc_ablation_diagnostics.to_csv(
    "/content/wtmc_structural_ablation_diagnostics.csv",
    index=False,
)

print("\nSaved:")
print("/content/wtmc_structural_ablation_baselines.csv")
print("/content/wtmc_structural_ablation_diagnostics.csv")


WTMC2021 — STRUCTURAL ABLATION BASELINES
Calibration thresholds reused unchanged:
  COUNT P99 = 12075.940000
  RATE  P99 = 5.611613
  REP   P99 = 715.509231

Environment          Model     Truth_Definition  Flagged  TP  FP  FN  TN  Precision  Recall       F1      FPR      Lift  Unique_Flagged_Sources  Unique_TP_Sources
   WTMC2021 COUNT_ONLY_P99 AttackShare_GE_50pct        9   5   4  11 573   0.555556  0.3125 0.400000 0.006932 20.590278                       4                  1
   WTMC2021  RATE_ONLY_P99 AttackShare_GE_50pct        7   5   2  11 575   0.714286  0.3125 0.434783 0.003466 26.473214                       2                  1
   WTMC2021   REP_ONLY_P99 AttackShare_GE_50pct       10  10   0   6 577   1.000000  0.6250 0.769231 0.000000 37.062500                       1                  1
   WTMC2021           S_BV AttackShare_GE_50pct        6   5   1  11 576   0.833333  0.3125 0.454545 0.001733 30.885417                       2                  1
   WTMC2021          S_REP 

In [ ]:
# ============================================================
# COMPLETE COLAB RUNTIME ENVIRONMENT SNAPSHOT
# Saves all reproducibility information into ONE text file.
# ============================================================

import sys
import os
import platform
import subprocess
import datetime
import importlib
import psutil

OUTPUT_FILE = "/content/runtime_environment_full.txt"

def run_cmd(cmd):
    try:
        return subprocess.check_output(
            cmd,
            shell=True,
            stderr=subprocess.STDOUT,
            text=True
        ).strip()
    except Exception as e:
        return f"[Command failed: {e}]"

def pkg_version(name):
    try:
        module = importlib.import_module(name)
        return getattr(module, "__version__", "version attribute unavailable")
    except Exception as e:
        return f"not available ({e})"

now = datetime.datetime.now(datetime.timezone.utc)

vm = psutil.virtual_memory()
disk = psutil.disk_usage("/")

important_packages = [
    "numpy",
    "pandas",
    "scipy",
    "sklearn",
    "duckdb",
    "networkx",
    "matplotlib",
    "pyarrow",
    "psutil"
]

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:

    f.write("=" * 80 + "\n")
    f.write("COLAB RUNTIME ENVIRONMENT SNAPSHOT\n")
    f.write("=" * 80 + "\n\n")

    f.write("Captured (UTC): "
            + now.strftime("%Y-%m-%d %H:%M:%S UTC") + "\n\n")

    # --------------------------------------------------------
    # Python
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("PYTHON\n")
    f.write("=" * 80 + "\n")

    f.write(f"Python version:\n{sys.version}\n\n")
    f.write(f"Python executable:\n{sys.executable}\n\n")
    f.write(f"Implementation:\n{platform.python_implementation()}\n\n")

    # --------------------------------------------------------
    # Operating system
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("OPERATING SYSTEM\n")
    f.write("=" * 80 + "\n")

    f.write(f"platform.platform():\n{platform.platform()}\n\n")
    f.write(run_cmd("cat /etc/os-release") + "\n\n")
    f.write("Kernel:\n")
    f.write(run_cmd("uname -a") + "\n\n")

    # --------------------------------------------------------
    # CPU
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("CPU\n")
    f.write("=" * 80 + "\n")

    f.write(run_cmd("lscpu") + "\n\n")
    f.write(f"Logical CPU count: {psutil.cpu_count(logical=True)}\n")
    f.write(f"Physical CPU count: {psutil.cpu_count(logical=False)}\n\n")

    # --------------------------------------------------------
    # RAM
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("MEMORY\n")
    f.write("=" * 80 + "\n")

    f.write(f"Total RAM:     {vm.total / (1024**3):.2f} GB\n")
    f.write(f"Available RAM: {vm.available / (1024**3):.2f} GB\n")
    f.write(f"Used RAM:      {vm.used / (1024**3):.2f} GB\n")
    f.write(f"RAM usage:     {vm.percent:.1f}%\n\n")

    # --------------------------------------------------------
    # Disk
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("DISK\n")
    f.write("=" * 80 + "\n")

    f.write(f"Total disk: {disk.total / (1024**3):.2f} GB\n")
    f.write(f"Used disk:  {disk.used / (1024**3):.2f} GB\n")
    f.write(f"Free disk:  {disk.free / (1024**3):.2f} GB\n\n")

    # --------------------------------------------------------
    # GPU
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("GPU / ACCELERATOR\n")
    f.write("=" * 80 + "\n")

    gpu_info = run_cmd(
        "nvidia-smi --query-gpu="
        "name,memory.total,driver_version "
        "--format=csv,noheader 2>/dev/null"
    )

    if gpu_info:
        f.write(gpu_info + "\n\n")
        f.write("Full NVIDIA-SMI:\n")
        f.write(run_cmd("nvidia-smi") + "\n\n")
    else:
        f.write("No NVIDIA GPU detected or nvidia-smi unavailable.\n\n")

    # --------------------------------------------------------
    # Important package versions
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("IMPORTANT PACKAGE VERSIONS\n")
    f.write("=" * 80 + "\n")

    for package in important_packages:
        f.write(f"{package}: {pkg_version(package)}\n")

    f.write("\n")

    # --------------------------------------------------------
    # Google Colab
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("GOOGLE COLAB\n")
    f.write("=" * 80 + "\n")

    try:
        import google.colab
        f.write("Environment: Google Colab\n")
        f.write(
            "google.colab version: "
            + str(getattr(google.colab, "__version__", "not exposed"))
            + "\n"
        )
    except Exception:
        f.write("Google Colab module not detected.\n")

    f.write("\n")

    # --------------------------------------------------------
    # Environment variables relevant to computation
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("SELECTED ENVIRONMENT VARIABLES\n")
    f.write("=" * 80 + "\n")

    selected_env = [
        "COLAB_RELEASE_TAG",
        "COLAB_BACKEND_VERSION",
        "CUDA_VERSION",
        "LANG",
        "TZ"
    ]

    for var in selected_env:
        f.write(f"{var}: {os.environ.get(var, 'not set')}\n")

    f.write("\n")

    # --------------------------------------------------------
    # Complete pip package snapshot
    # --------------------------------------------------------
    f.write("=" * 80 + "\n")
    f.write("COMPLETE PIP FREEZE\n")
    f.write("=" * 80 + "\n")

    f.write(run_cmd(f'"{sys.executable}" -m pip freeze') + "\n")

    f.write("\n" + "=" * 80 + "\n")
    f.write("END OF ENVIRONMENT SNAPSHOT\n")
    f.write("=" * 80 + "\n")

print("=" * 80)
print("Environment snapshot saved successfully.")
print(f"File: {OUTPUT_FILE}")
print(f"Size: {os.path.getsize(OUTPUT_FILE) / 1024:.1f} KB")
print("=" * 80)

# Optional: automatically download the file from Colab
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except Exception:
    print("Automatic download unavailable; download the file manually from /content.")

Environment snapshot saved successfully.
File: /content/runtime_environment_full.txt
Size: 18.8 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

for f in os.listdir("/content"):
    if "ablation" in f.lower():
        print(f)

wtmc_structural_ablation_diagnostics.csv
wtmc_structural_ablation_baselines.csv


In [ ]:
from google.colab import files

files.download('/content/wtmc_structural_ablation_baselines.csv')
files.download('/content/wtmc_structural_ablation_diagnostics.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>